# Introduction to Quantum Computing with Qiskit

**Lecture Exercises — Quantum Computing**

This notebook is an introduction to quantum computing using IBM's open-source framework ``Qiskit``. We will cover:

1. Qubits, state vectors, and the Bloch sphere
2. Single-qubit gates
3. Measurement and simulation
4. Multi-qubit systems and entanglement
5. Quantum teleportation protocol
6. Exercises


### References
- Nielsen & Chuang, *Quantum Computation and Quantum Information*
- [Qiskit documentation](https://docs.quantum.ibm.com/)

## 1. Setup and Imports

First, install and import the packages we need. You can find the packages needed for this notebook in the file `requirements.txt` in the repository.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit.quantum_info import Statevector, Operator

## 2. Qubits, State Vectors, and the Bloch Sphere

### The qubit

A classical bit is either $0$ or $1$. A qubit can exist in a superposition of the computational basis states:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad \alpha,\beta \in \mathbb{C}, \quad |\alpha|^2 + |\beta|^2 = 1.$$

In the computational basis, we represent these as column vectors:

$$|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \qquad |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}.$$

### The Bloch sphere

Any single-qubit pure state can be parameterised (up to global phase) as:

$$|\psi\rangle = \cos\frac{\theta}{2}|0\rangle + e^{i\phi}\sin\frac{\theta}{2}|1\rangle$$

where $\theta \in [0,\pi]$ and $\phi \in [0, 2\pi)$. This maps every qubit state to a point on the Bloch sphere.

Below, we will visualise the quantum states using the `Statevector` operation, which manipulates the statevector of the quantum circuit.

In [ ]:
# |0> state
sv_zero = Statevector.from_label('0')
print("|0> state vector:", sv_zero)
plot_bloch_multivector(sv_zero)

In [ ]:
# |1> state 
sv_one = Statevector.from_label('1')
print("|1> state vector:", sv_one)
plot_bloch_multivector(sv_one)

In [ ]:
# |+> state 
sv_plus = Statevector.from_label('+')
print("|+> state vector:", sv_plus)
plot_bloch_multivector(sv_plus)

In [ ]:
# Custom state: (1/sqrt(2))(|0> + i|1>) = |+i> (the |R> state)
sv_custom = Statevector([1/np.sqrt(2), 1j/np.sqrt(2)])
print("Custom state vector:", sv_custom)
plot_bloch_multivector(sv_custom)

## 3. Single-Qubit Gates

Quantum gates are unitary operators acting on qubits. Since the state space of a single qubit is $\mathbb{C}^2$, single-qubit gates are $2\times 2$ unitary matrices.

### 3.1 The Pauli gates

$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \quad Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}, \quad Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

- **X** (bit-flip): $X|0\rangle = |1\rangle$, $X|1\rangle = |0\rangle$. Analogous to a classical NOT gate.
- **Z** (phase-flip): $Z|0\rangle = |0\rangle$, $Z|1\rangle = -|1\rangle$.
- **Y** $= iXZ$

### 3.2 The Hadamard gate

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & \phantom{-}1 \\ 1 & -1 \end{pmatrix}$$

Creates superpositions: 

$$H|0\rangle = |+\rangle = \frac{1}{\sqrt{2}} \left( \vert 0 \rangle + \vert 1 \rangle\right),$$

$$H|1\rangle = |-\rangle = \frac{1}{\sqrt{2}} \left( \vert 0 \rangle - \vert 1 \rangle\right),$$

### 3.3 Phase gates

$$S = \begin{pmatrix} 1 & 0 \\ 0 & i \end{pmatrix}, \qquad T = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\pi/4} \end{pmatrix}$$

Note that $S = T^2$ and $Z = S^2$.

### 3.4 Rotation gates

$$R_x(\theta) = e^{-i\theta X/2} = \begin{pmatrix} \cos\frac{\theta}{2} & -i\sin\frac{\theta}{2} \\ -i\sin\frac{\theta}{2} & \cos\frac{\theta}{2} \end{pmatrix}$$

and analogously $R_y(\theta)$, $R_z(\theta)$. Any single-qubit unitary can be decomposed as $U = e^{i\alpha} R_z(\beta) R_y(\gamma) R_z(\delta)$.

### Building and drawing circuits

Let's now build circuits with these gates.

In [ ]:
### Demonstrate Pauli-X gate (bit flip)
qc = QuantumCircuit(1)      # Builds a circuit with a single qubit
qc.x(0)                     # Applies gate to qubit 0
qc.draw('mpl')              # Draw's the circuit

In [ ]:
# Check the action: X|0> = |1>
sv = Statevector.from_label('0')
sv = sv.evolve(qc)
print("After X gate:", sv)
plot_bloch_multivector(sv)

In [ ]:
# Hadamard gate: H|0> = |+>
qc_h = QuantumCircuit(1)
qc_h.h(0)

sv = Statevector.from_label('0').evolve(qc_h)
print("After H gate:", sv)
plot_bloch_multivector(sv)

In [ ]:
# Compose several gates and inspect the unitary matrix
qc_composed = QuantumCircuit(1)
qc_composed.h(0)
qc_composed.t(0)
qc_composed.h(0)

# Extract the unitary matrix
op = Operator(qc_composed)                  # Converts quantum circuit into a unitary operation
print("Unitary matrix of H-T-H circuit:")
print(np.array(op).round(4))
print()
qc_composed.draw('mpl')

In [ ]:
# Rotation gates: Rx(pi) is equivalent to X (up to global phase)
print("Rx(pi) matrix:")
qc_rx = QuantumCircuit(1)
qc_rx.rx(np.pi, 0)
print(np.array(Operator(qc_rx)).round(4))

print("\nX gate matrix:")
print(Operator.from_label('X').to_matrix())

print("\nThey differ by a global phase of -i, which is physically unobservable.")

## 4. Measurement and Simulation

When we measure a qubit in state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ in the computational basis, we obtain outcome $0$ with probability $|\alpha|^2$ and outcome $1$ with probability $|\beta|^2$. The state collapses to the measured outcome.

### Simulating circuits with Qiskit Aer

We can emulate quantum circuits using the `AerSimulator` backend:
- `AerSimulator()` — simulates with sampling (emulates a real device)
- `AerSimulator(method='statevector')` — gives the full state vector (equivalent to the `Statevector` operator)

In [ ]:
# Build a simple circuit: put qubit in |+> and measure
qc = QuantumCircuit(1, 1)   # Builds circuit with 1 qubit, 1 classical bit
qc.h(0)
qc.measure(0, 0)            # Measures qubit 0 to classical bit 0
qc.draw('mpl')

In [ ]:
# Simulate with 1024 shots (N.B. 1024 is the default number of shots)
simulator = AerSimulator()                              # Load simulator instance
compiled = transpile(qc, simulator)                     # transpile circuit
result = simulator.run(compiled, shots=1024).result()

counts = result.get_counts()
print("Measurement results:", counts)
plot_histogram(counts)

As expected, we get roughly 50/50 outcomes for $|0\rangle$ and $|1\rangle$, since $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$.

Note: the results are stochastic: run the cell again and you'll get slightly different counts.

Note: that the circuit does not strictly need to be transpiled here, since the AerSimulator can emulate any gate. However, when running on the real device, one would need to transpile into the gates that can be implemented on the device.

In [ ]:
# Statevector simulation (no measurement collapse -- gives exact amplitudes)
qc_sv = QuantumCircuit(1)
qc_sv.h(0)

sv_sim = AerSimulator(method='statevector')     # Load in statevector instance
qc_sv.save_statevector()                        # Tell the simulator to save the state
compiled = transpile(qc_sv, sv_sim)
result = sv_sim.run(compiled).result()
statevector = result.get_statevector()
print("Exact state vector:", statevector)

In [ ]:
# We can also extract probabilities directly from a Statevector object
sv = Statevector.from_label('0')
qc_temp = QuantumCircuit(1)
qc_temp.h(0)
sv = sv.evolve(qc_temp)

probs = sv.probabilities_dict()
print("Probabilities:", probs)

## 5. Multi-Qubit Systems and Entanglement

### Tensor products

The state space of an $n$-qubit system is the tensor product $(\mathbb{C}^2)^{\otimes n}$, which has dimension $2^n$. A two-qubit system lives in $\mathbb{C}^4$ with computational basis $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$.

A product state is one that can be written as $|\psi\rangle \otimes |\phi\rangle$. An entangled state is one that *cannot* be factored this way.

### The CNOT gate

The **controlled-NOT** (CNOT or CX) gate is the most common two-qubit gate:

$$\text{CNOT} = |0\rangle\langle 0| \otimes I + |1\rangle\langle 1| \otimes X = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}$$

It flips the target qubit if and only if the control qubit is $|1\rangle$, such that

$$
\text{CNOT} \vert 00 \rangle = \vert 00 \rangle~, \qquad \text{CNOT} \vert 10 \rangle = \vert 11 \rangle~,\\
\text{CNOT} \vert 01 \rangle = \vert 01 \rangle~, \qquad \text{CNOT} \vert 11 \rangle = \vert 10 \rangle~.
$$

The CNOT gate is the only two-qubit gate needed to implement a universal computational gate set. 

### Bell states

The four Bell states form a maximally entangled basis for two qubits:

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle), \qquad |\Phi^-\rangle = \frac{1}{\sqrt{2}}(|00\rangle - |11\rangle)$$
$$|\Psi^+\rangle = \frac{1}{\sqrt{2}}(|01\rangle + |10\rangle), \qquad |\Psi^-\rangle = \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)$$

In [ ]:
# Create the Bell state |Phi+> = (|00> + |11>)/sqrt(2)
bell = QuantumCircuit(2)
bell.h(0)                   # Apply H gate to put first qubit in superposition
bell.cx(0, 1)               # CNOT: entangle the two qubits
bell.draw('mpl')

In [ ]:
# Inspect the state vector
sv_bell = Statevector.from_label('00').evolve(bell)
print("Bell state |Phi+>:")
print(sv_bell)
print("\nProbabilities:", sv_bell.probabilities_dict())
plot_bloch_multivector(sv_bell)

Note that the individual qubit Bloch vectors have zero length. This is the signature of maximal entanglement. Neither qubit has a well-defined individual state; the quantum information is entirely in the correlations between them. You can test this by removing the entanglement between the qubits. 

We can now run this circuit using the emulator and measuring the circuit.

In [ ]:
# Measure the Bell state
bell_meas = QuantumCircuit(2, 2)        # Circuit with 2 qubits and 2 classical bits
bell_meas.h(0)                          # Apply Hadamard to qubit 0
bell_meas.cx(0, 1)                      # Apply CNOT gate across qubits 0 and 1
bell_meas.measure([0, 1], [0, 1])       # Measure qubits [0,1] to classical bits [0,1]

simulator = AerSimulator()              
compiled = transpile(bell_meas, simulator)
result = simulator.run(compiled, shots=2000).result()
counts = result.get_counts()
print("Bell state measurements:", counts)
plot_histogram(counts)

We only ever measure $|00\rangle$ or $|11\rangle$. The outcomes are perfectly correlated. This is the hallmark of entanglement.

### Creating all four Bell states

Starting from $|00\rangle$ and applying $H \otimes I$ then CNOT gives $|\Phi^+\rangle$. We obtain the other three by preparing different input states:

| Input state | Bell state |
|---|---|
| $\vert 00 \rangle$ | $\vert \Phi^+\rangle$ |
| $\vert 10 \rangle$ | $\vert \Phi^-\rangle$ |
| $\vert 01 \rangle$ | $\vert \Psi^+\rangle$ |
| $\vert 11 \rangle$ | $\vert \Psi^-\rangle$ |


Note this is slightly different in the code below. This is because Qiskit bitstrings are read "backwards".

In [ ]:
# Create all four Bell states
labels = ['Phi+', 'Phi-', 'Psi+', 'Psi-']
initial = ['00', '01', '10', '11']

for label, init in zip(labels, initial):
    sv = Statevector.from_label(init)
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)
    sv = sv.evolve(qc)
    print(f"|{label}> = {sv}")

## 6. Quantum Teleportation

Quantum teleportation allows Alice to transmit an unknown qubit state $|\psi\rangle$ to Bob, using:
1. A shared Bell pair (one qubit each)
2. Two classical bits of communication

This does not violate the no-cloning theorem (Alice's qubit is destroyed) nor allow faster-than-light communication (the classical bits must be sent).

### The protocol

1. Alice and Bob share $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$.
2. Alice applies CNOT (her message qubit as control, her Bell qubit as target).
3. Alice applies $H$ to her message qubit.
4. Alice measures both her qubits, obtaining two classical bits $(m_1, m_2)$.
5. Alice sends $(m_1, m_2)$ to Bob classically.
6. Bob applies corrections: if $m_2 = 1$, apply $X$; if $m_1 = 1$, apply $Z$.
7. Bob's qubit is now in state $|\psi\rangle$.

In [ ]:
def teleportation_circuit():
    '''Build the quantum teleportation circuit.
    
    Qubit 0: Alice message qubit (the state to teleport)
    Qubit 1: Alice half of the Bell pair
    Qubit 2: Bob half of the Bell pair
    '''
    qc = QuantumCircuit(3, 2)               # Build circuit with 3 qubits and 2 bits
    
    # Step 0: Prepare the state to teleport on qubit 0
    qc.rx(np.pi/4, 0)                       # Prepare |psi> = Rx(pi/4)|0>
    qc.barrier(label='prep')                # NB: Barriers have no effect for the emulator
                                            #     but alter transpilation in real devices
    
    # Step 1: Create Bell pair between qubits 1 and 2
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier(label='Bell pair')
    
    # Step 2: Alice's operations
    qc.cx(0, 1)                             # CNOT with message qubit as control
    qc.h(0)                                 # Hadamard on message qubit
    qc.barrier(label='Alice')
    
    # Step 3: Alice measures her two qubits
    qc.measure(0, 0)
    qc.measure(1, 1)
    qc.barrier(label='measure')
    
    # Step 4: Bob applies corrections conditioned on classical bits
    # In Qiskit 1.x, we use if_test for classical conditioning
    with qc.if_test((1, 1)):   # If classical bit 1 is 1, apply X
        qc.x(2)
    with qc.if_test((0, 1)):   # If classical bit 0 is 1, apply Z
        qc.z(2)
    
    return qc

qc_teleport = teleportation_circuit()
qc_teleport.draw('mpl', fold=-1)

In [ ]:
# What state are we trying to teleport?
qc_target = QuantumCircuit(1)
qc_target.rx(np.pi/4, 0)
target_state = Statevector.from_label('0').evolve(qc_target)
print("State to teleport:")
print(target_state)
print("Probabilities:", target_state.probabilities_dict())

We now look to verify that the circuit is doing what is expected. We can do this by using a further classical bit to measure Bob's qubit

In [ ]:
# Simulate the teleportation and verify Bob receives the correct state
qc_verify = QuantumCircuit(3, 3)

# Prepare the state to teleport
qc_verify.rx(np.pi/4, 0)
qc_verify.barrier()

# Bell pair
qc_verify.h(1)
qc_verify.cx(1, 2)
qc_verify.barrier()

# Alice's operations
qc_verify.cx(0, 1)
qc_verify.h(0)
qc_verify.barrier()

# Alice measures
qc_verify.measure(0, 0)
qc_verify.measure(1, 1)
qc_verify.barrier()

# Bob's corrections (using if_test for Qiskit 2.x)
with qc_verify.if_test((1, 1)):
    qc_verify.x(2)
with qc_verify.if_test((0, 1)):
    qc_verify.z(2)

# Measure Bob's qubit
qc_verify.measure(2, 2)

shots = 10000
simulator = AerSimulator()
compiled = transpile(qc_verify, simulator)
result = simulator.run(compiled, shots=shots).result()
counts = result.get_counts()

# Extract just Bob's measurement outcomes (bit 2)
bob_counts = {'0': 0, '1': 0}
for outcome, count in counts.items():
    bob_bit = outcome[0]  # In Qiskit, bit order is reversed: bit 2 is leftmost
    bob_counts[bob_bit] += count

total = sum(bob_counts.values())
print(f'Bob outcomes: {bob_counts}')
print(f'P(0) = {(bob_counts["0"])/shots}  (expected: {np.cos(np.pi/8)**2:.4f})')
print(f'P(1) = {(bob_counts["1"])/shots}  (expected: {np.sin(np.pi/8)**2:.4f})')
plot_histogram(bob_counts)

The measurement probabilities on Bob’s qubit match those of the target state, $|\psi\rangle = R_x(\pi/4)|0\rangle$. This confirms the populations are correct, but not the full quantum state. To verify full teleportation, one should compare the final statevector or perform tomography.

### 8. Fidelity - Verifying the teleported state with the SWAP test

Measuring Bob's qubit in the computational basis only checks the populations of the teleported state, not its relative phase. To compare the full quantum state, we can use the SWAP test.

For two pure states $\lvert \psi \rangle$ and $\lvert \phi \rangle$, the SWAP test estimates the fidelity
$$
F = |\langle \psi | \phi \rangle|^2.
$$
If the ancilla is measured in state $\lvert 0 \rangle$, then
$$
P(0) = \frac{1 + F}{2},
$$
so that
$$
F = 2P(0) - 1.
$$

If the teleportation has worked perfectly, then the two states should be identify and we should find a fidelity close to 1. Below we check this by comparing two identical states:

In [ ]:
def prepare_rx_pi_over_4(qc, qubit):
    """Prepare the test state Rx(pi/4)|0> on the chosen qubit."""
    qc.rx(np.pi/4, qubit)

def swap_test_circuit():
    """
    Compare two copies of the same state using the SWAP test.
    Qubit 0: ancilla
    Qubit 1: state |psi>
    Qubit 2: state |phi>
    """
    qc = QuantumCircuit(3, 1)

    # Prepare the two states to compare
    prepare_rx_pi_over_4(qc, 1)   # |psi>
    prepare_rx_pi_over_4(qc, 2)   # |phi>
    
    # SWAP test
    qc.h(0)
    qc.cswap(0, 1, 2)
    qc.h(0)
    qc.measure(0, 0)

    return qc

qc = swap_test_circuit()
qc.draw("mpl")

We now estimate the fidelity of the two states:

In [ ]:
sim = AerSimulator()
result = sim.run(qc, shots=10000).result()
counts = result.get_counts()

p0 = counts.get('0', 0) / 10000
fidelity = 2 * p0 - 1

print("Counts:", counts)
print(f"P(ancilla = 0) = {p0:.4f}")
print(f"Estimated fidelity = {fidelity:.4f}")

You could now try combining the quantum teleportation script and the SWAP test script to measure the fidelity of the teleported state:

In [ ]:
###
 
### Your code here

###

## 7. Exercises

Work through the following exercises to test your understanding. Solution cells are provided at the bottom.


### Exercise 1: Exploring single-qubit gates

**(a)** Create a circuit that applies the sequence $Z \cdot H \cdot X$ to $|0\rangle$. What is the resulting state vector? Verify analytically and using Qiskit.

**(b)** Show (by computing the matrix or applying to all basis states) that $HZH = X$. This is an example of a useful identity: conjugation by Hadamard swaps $X \leftrightarrow Z$.

**(c)** The $S$ gate satisfies $S^2 = Z$. Verify this using the `Operator` class.

In [ ]:
# Exercise 1(a): Your code here
# Hint: Create a QuantumCircuit, apply gates, evolve a Statevector


In [ ]:
# Exercise 1(b): Your code here
# Hint: Build the circuit H-Z-H, extract its Operator, and compare with X


In [ ]:
# Exercise 1(c): Your code here


### Exercise 2: The GHZ state

The **Greenberger-Horne-Zeilinger (GHZ) state** for $n$ qubits is:

$$|\text{GHZ}\rangle = \frac{1}{\sqrt{2}}(|00\ldots 0\rangle + |11\ldots 1\rangle)$$

**(a)** Construct a circuit that creates the 3-qubit GHZ state $\frac{1}{\sqrt{2}}(|000\rangle + |111\rangle)$.

*Hint: Generalise the Bell state construction.*

**(b)** Simulate the circuit and verify you only measure $|000\rangle$ and $|111\rangle$.

**(c)** Generalise your circuit to $n = 5$ qubits.

In [ ]:
# Exercise 2(a): Your code here


In [ ]:
# Exercise 2(b): Your code here


In [ ]:
# Exercise 2(c): Your code here


### Exercise 3: Quantum phase kickback

Phase kickback is the mechanism behind many quantum algorithms (Deutsch-Jozsa, quantum phase estimation, Grover's algorithm).

Consider a controlled-$U$ gate where the target qubit is in an eigenstate $|u\rangle$ of $U$ with eigenvalue $e^{i\phi}$:

$$\text{C-}U \big(\alpha|0\rangle + \beta|1\rangle\big)|u\rangle = \big(\alpha|0\rangle + \beta e^{i\phi}|1\rangle\big)|u\rangle$$

The phase "kicks back" onto the control qubit!

**Exercise:** Demonstrate phase kickback with $U = T$ (which has eigenvalue $e^{i\pi/4}$ on $|1\rangle$):

1. Prepare the control qubit in $|+\rangle$ and the target in $|1\rangle$.
2. Apply a controlled-$T$ gate.
3. Show that the control qubit picks up a phase of $e^{i\pi/4}$ (i.e., its state becomes $\frac{1}{\sqrt{2}}(|0\rangle + e^{i\pi/4}|1\rangle)$).

In [ ]:
# Exercise 3: Your code here
# Hint: Use qc.cp(np.pi/4, control, target) for a controlled-phase gate


### Exercise 4: The Deutsch algorithm

The **Deutsch algorithm** (the simplest quantum algorithm) determines whether a function $f: \{0,1\} \to \{0,1\}$ is *constant* ($f(0)=f(1)$) or *balanced* ($f(0)\neq f(1)$) using a single query.

The circuit uses two qubits and an *oracle* $U_f$ that maps $|x\rangle|y\rangle \to |x\rangle|y \oplus f(x)\rangle$.

**Steps:**
1. Initialise $|0\rangle|1\rangle$.
2. Apply $H$ to both qubits.
3. Apply the oracle $U_f$.
4. Apply $H$ to the first qubit.
5. Measure the first qubit: $|0\rangle$ = constant, $|1\rangle$ = balanced.

**(a)** Implement the Deutsch algorithm for the balanced function $f(x) = x$ (the oracle is just a CNOT).

**(b)** Implement it for the constant function $f(x) = 0$ (the oracle is the identity — do nothing).

**(c)** Verify that measurement of the first qubit correctly identifies each case.

In [ ]:
def deutsch_algorithm(oracle_type='balanced'):
    '''Implement the Deutsch algorithm.
    
    Parameters:
        oracle_type: "balanced" for f(x)=x, "constant" for f(x)=0
    
    Returns:
        QuantumCircuit
    '''
    qc = QuantumCircuit(2, 1)
    
    # Your implementation here
    # ...
    
    return qc


### Exercise 5: Quantum state tomography (conceptual)

In experiment, we cannot directly read off $\alpha$ and $\beta$ from a qubit — we can only measure. **State tomography** reconstructs the state by measuring in multiple bases.

For a single qubit, we need measurements in three bases:
- **Z basis** (computational basis): measure directly
- **X basis**: apply $H$ then measure
- **Y basis**: apply $S^\dagger H$ then measure

**(a)** Prepare the state $|\psi\rangle = R_y(\pi/3)|0\rangle$.

**(b)** Write three circuits that measure this state in the Z, X, and Y bases respectively. Run each with 4096 shots.

**(c)** From the measurement statistics, estimate $\langle X \rangle$, $\langle Y \rangle$, $\langle Z \rangle$ (the Bloch vector components). Compare with the exact values.

*Hint:* For a measurement with outcomes $\{0, 1\}$ with counts $\{n_0, n_1\}$:

$$\langle O \rangle \approx \frac{n_0 - n_1}{n_0 + n_1}$$

where $O$ is the Pauli operator corresponding to the measurement basis.

In [ ]:
# Exercise 5: Your code here


---

### Solutions

Below are solutions to the exercises. Try them yourself first!

#### Solution to Exercise 1

<details>
<summary>Solution 1(a)</summary>

```python
# 1(a) Z.H.X |0>
qc = QuantumCircuit(1)
qc.x(0)
qc.h(0)
qc.z(0)

sv = Statevector.from_label('0').evolve(qc)
print("Z.H.X |0> =", sv)
print()

# Analytical: X|0> = |1>, H|1> = |-> = (|0>-|1>)/sqrt(2),
# Z|-> = -(|0>+|1>)/sqrt(2) = -|+>
# Global phase is unobservable, so this is equivalent to |+>
print("Probabilities:", sv.probabilities_dict())

<details>
<summary>Solution 1(b)</summary>

```python
# 1(b) Show HZH = X
qc_hzh = QuantumCircuit(1)
qc_hzh.h(0)
qc_hzh.z(0)
qc_hzh.h(0)

op_hzh = Operator(qc_hzh)
op_x = Operator.from_label('X')

print("HZH matrix:")
print(np.array(op_hzh).round(4))
print("\nX matrix:")
print(op_x.to_matrix())
print(f"\nHZH == X (up to global phase)? {op_hzh.equiv(op_x)}")

<details>
<summary>Solution 1(c)</summary>

```python
# 1(c) Verify S^2 = Z
op_s = Operator.from_label('S')
op_s2 = op_s.compose(op_s)  # S.S
op_z = Operator.from_label('Z')

print("S^2 matrix:")
print(np.array(op_s2).round(4))
print("\nZ matrix:")
print(op_z.to_matrix())
print(f"\nS^2 == Z? {op_s2.equiv(op_z)}")

#### Solution to Exercise 2

<details>
<summary>Solution 2(a)</summary>

```python
# 2(a) 3-qubit GHZ state
ghz3 = QuantumCircuit(3)
ghz3.h(0)
ghz3.cx(0, 1)
ghz3.cx(0, 2)

sv_ghz = Statevector.from_label('000').evolve(ghz3)
print("3-qubit GHZ state:", sv_ghz)
print("Probabilities:", sv_ghz.probabilities_dict())
ghz3.draw('mpl')

<details>
<summary>Solution 2(b)</summary>

```python
# 2(b) Simulate and verify
ghz3_meas = ghz3.copy()
ghz3_meas.add_register(ClassicalRegister(3))
ghz3_meas.measure([0, 1, 2], [0, 1, 2])

simulator = AerSimulator()
compiled = transpile(ghz3_meas, simulator)
result = simulator.run(compiled, shots=2048).result()
counts = result.get_counts()
print("GHZ measurements:", counts)
plot_histogram(counts)

<details>
<summary>Solution 2(c)</summary>

```python
# 2(c) 5-qubit GHZ
n = 5
ghz5 = QuantumCircuit(n)
ghz5.h(0)
for i in range(1, n):
    ghz5.cx(0, i)

sv_ghz5 = Statevector.from_label('0' * n).evolve(ghz5)
print(f"{n}-qubit GHZ state probabilities:", sv_ghz5.probabilities_dict())
ghz5.draw('mpl')

#### Solution to Exercise 3

<details>
<summary>Solution 3</summary>

```python
# Phase kickback with controlled-T
qc = QuantumCircuit(2)
qc.h(0)        # Control qubit in |+>
qc.x(1)        # Target qubit in |1>
qc.cp(np.pi/4, 0, 1)  # Controlled-T (controlled-phase with angle pi/4)

sv = Statevector.from_label('00').evolve(qc)
print("State after controlled-T:")
print(sv)
print("\nProbabilities:", sv.probabilities_dict())

# The control qubit should be in (|0> + e^{i*pi/4}|1>)/sqrt(2)
# while the target remains in |1>
print("\nExpected: (1/sqrt(2))(|01> + e^{i*pi/4}|11>)")
print(f"Expected amplitudes: |01> -> {1/np.sqrt(2):.4f}, |11> -> {np.exp(1j*np.pi/4)/np.sqrt(2):.4f}")

#### Solution to Exercise 4

<details>
<summary>Solution 4</summary>

```python
def deutsch_algorithm(oracle_type='balanced'):
    '''Implement the Deutsch algorithm.'''
    qc = QuantumCircuit(2, 1)
    
    # Step 1: Initialise |01>
    qc.x(1)
    
    # Step 2: Apply H to both qubits
    qc.h(0)
    qc.h(1)
    qc.barrier()
    
    # Step 3: Apply oracle
    if oracle_type == 'balanced':
        # f(x) = x: oracle is CNOT
        qc.cx(0, 1)
    elif oracle_type == 'constant':
        # f(x) = 0: oracle is identity (do nothing)
        pass
    qc.barrier()
    
    # Step 4: Apply H to first qubit
    qc.h(0)
    
    # Step 5: Measure first qubit
    qc.measure(0, 0)
    
    return qc

# Test both cases
simulator = AerSimulator()

for oracle in ['balanced', 'constant']:
    qc = deutsch_algorithm(oracle)
    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=1024).result()
    counts = result.get_counts()
    print(f"Oracle: {oracle:10s} -> Measurement: {counts}")
    display(qc.draw('mpl'))

#### Solution to Exercise 5

<details>
<summary>Solution 5</summary>

```python
# State tomography of Ry(pi/3)|0>
theta = np.pi / 3
shots = 4096
simulator = AerSimulator()

# Z-basis measurement
qc_z = QuantumCircuit(1, 1)
qc_z.ry(theta, 0)
qc_z.measure(0, 0)

# X-basis measurement
qc_x = QuantumCircuit(1, 1)
qc_x.ry(theta, 0)
qc_x.h(0)
qc_x.measure(0, 0)

# Y-basis measurement
qc_y = QuantumCircuit(1, 1)
qc_y.ry(theta, 0)
qc_y.sdg(0)  # S-dagger
qc_y.h(0)
qc_y.measure(0, 0)

results = {}
for label, qc in [('Z', qc_z), ('X', qc_x), ('Y', qc_y)]:
    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=shots).result()
    counts = result.get_counts()
    n0 = counts.get('0', 0)
    n1 = counts.get('1', 0)
    expectation = (n0 - n1) / (n0 + n1)
    results[label] = expectation

print("Estimated Bloch vector from tomography:")
print(f"  <X> = {results['X']:.4f}")
print(f"  <Y> = {results['Y']:.4f}")
print(f"  <Z> = {results['Z']:.4f}")

print("\nExact values:")
print(f"  <X> = {np.sin(theta):.4f}")
print(f"  <Y> = {0.0:.4f}")
print(f"  <Z> = {np.cos(theta):.4f}")